# SARIMAX - 02: Validacao Walk-Forward
**Grupo 1 | Bitcoin diário**

Avalia a ordem selecionada por `auto_arima` no notebook 01: SARIMAX `(0,0,1) × (0,0,0,7)`, com intercepto e 51 features exógenas. A primeira previsão usa somente os 70% iniciais; a cada origem seguinte, o alvo recém-observado é incorporado ao estado sem reajustar os parâmetros. Os 30% finais são usados exclusivamente para avaliação.

In [1]:
# SARIMAX - 02: dados e ajuste inicial
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
SARIMAX_ORDER = (0, 0, 1)
SARIMAX_SEASONAL_ORDER = (0, 0, 0, 7)
MAXITER = 50

cwd = Path.cwd().resolve()
root_candidates = list(cwd.parents) + [cwd]
root_candidates += [parent / 'analyses' / 'grupo1' for parent in [cwd, *cwd.parents]]
GROUP_ROOT = next(
    (candidate for candidate in dict.fromkeys(root_candidates)
     if (candidate / '01_dados' / 'feature_metadata.json').is_file()),
    None
)
if GROUP_ROOT is None:
    raise FileNotFoundError('Nao foi possivel localizar analyses/grupo1/01_dados/feature_metadata.json.')

DATA_DIR = GROUP_ROOT / '01_dados'
OUTPUT_DIR = GROUP_ROOT / '02_modelos' / 'SARIMAX'
with (DATA_DIR / 'feature_metadata.json').open('r', encoding='utf-8') as file:
    metadata = json.load(file)
FEATURE_COLS = metadata['feature_cols']
TARGET_COL = metadata['target_col']
train = pd.read_csv(DATA_DIR / 'bitcoin_features_train.csv', parse_dates=['date']).sort_values('date').set_index('date')
test = pd.read_csv(DATA_DIR / 'bitcoin_features_test.csv', parse_dates=['date']).sort_values('date').set_index('date')
if not train.index.to_series().diff().dropna().eq(pd.Timedelta(days=1)).all():
    raise ValueError('O treino nao esta regularizado em frequencia diaria.')
if not test.index.to_series().diff().dropna().eq(pd.Timedelta(days=1)).all():
    raise ValueError('O teste nao esta regularizado em frequencia diaria.')
X_train = train[FEATURE_COLS].astype(float)
y_train = train[TARGET_COL].astype(float)
X_test = test[FEATURE_COLS].astype(float)
y_test = test[TARGET_COL].astype(float)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_COLS)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_COLS)
y_train_model = y_train.reset_index(drop=True)
y_test_model = y_test.reset_index(drop=True)

sarimax_fit = SARIMAX(
    endog=y_train_model,
    exog=X_train_scaled,
    order=SARIMAX_ORDER,
    seasonal_order=SARIMAX_SEASONAL_ORDER,
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False, maxiter=MAXITER)

print(f'Raiz do Grupo 1: {GROUP_ROOT}')
print(f'Treino: {len(train)} observacoes | Teste: {len(test)} observacoes')
print(f'Features exogenas: {len(FEATURE_COLS)}')
print(f'Ordens SARIMAX: {SARIMAX_ORDER} x {SARIMAX_SEASONAL_ORDER} | intercepto: sim')
print(f'Convergiu: {sarimax_fit.mle_retvals.get("converged", "nao informado")}')

Raiz do Grupo 1: C:\Users\raqueltolomei-ieg\OneDrive - Instituto Germinare\Área de Trabalho\3 ano\Séries Temporais\PLSRegretion\pls-regration-comparation\analyses\grupo1
Treino: 1996 observacoes | Teste: 856 observacoes
Features exogenas: 51
Ordens SARIMAX: (0, 0, 1) x (0, 0, 0, 7) | intercepto: sim
Convergiu: True


In [2]:
# Previsoes 1 passo a frente; parametros fixos e estado atualizado sequencialmente.
from time import perf_counter

started_at = perf_counter()
predictions = []
state = sarimax_fit

for step in range(len(X_test)):
    next_index = pd.RangeIndex(len(y_train) + step, len(y_train) + step + 1)
    exog_step = X_test_scaled.iloc[[step]].set_axis(next_index)
    target_step = y_test_model.iloc[[step]].set_axis(next_index)
    prediction = state.get_forecast(steps=1, exog=exog_step).predicted_mean.iloc[0]
    predictions.append(float(prediction))
    state = state.append(endog=target_step, exog=exog_step, refit=False)

elapsed_seconds = perf_counter() - started_at
residuals = y_test.to_numpy() - np.asarray(predictions)
df_wf = pd.DataFrame({
    'date': test.index + pd.Timedelta(days=1),
    'target_date': test.index + pd.Timedelta(days=1),
    'priceClose_real': y_test.to_numpy(),
    'priceClose_pred_SARIMAX': predictions,
    'residuo_SARIMAX': residuals
})

mae = mean_absolute_error(y_test, predictions)
rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
r2 = r2_score(y_test, predictions)
mape = float(np.mean(np.abs(residuals / y_test.to_numpy())) * 100)
bias = float(np.mean(residuals))

print(f'Previsoes fora da amostra: {len(df_wf)}')
print(f'MAE: USD {mae:,.2f} | RMSE: USD {rmse:,.2f} | MAPE: {mape:.2f}% | R2: {r2:.4f}')
print(f'Vies medio (real - previsto): USD {bias:,.2f}')
print(f'Tempo walk-forward: {elapsed_seconds:.2f} s ({elapsed_seconds / len(df_wf) * 1000:.2f} ms/previsao)')
display(df_wf.head())

Previsoes fora da amostra: 856
MAE: USD 1,590.77 | RMSE: USD 2,130.91 | MAPE: 1.90% | R2: 0.9873
Vies medio (real - previsto): USD 693.86
Tempo walk-forward: 38.41 s (44.87 ms/previsao)


,date,target_date,priceClose_real,priceClose_pred_SARIMAX,residuo_SARIMAX
0,2024-05-08,2024-05-09,63049.959257,60415.681464,2634.277793
1,2024-05-09,2024-05-10,60792.776791,62670.270384,-1877.493594
2,2024-05-10,2024-05-11,60793.709599,59938.661407,855.048192
3,2024-05-11,2024-05-12,61448.394672,60627.792933,820.601739
4,2024-05-12,2024-05-13,62901.447455,60621.911692,2279.535763


In [3]:
export_cols = ['date', 'priceClose_real', 'priceClose_pred_SARIMAX', 'residuo_SARIMAX']
df_wf[export_cols].to_csv(OUTPUT_DIR / 'sarimax_previsoes_walkforward.csv', index=False)
metricas_export = {
    'modelo': 'SARIMAX',
    'base': 'bitcoin',
    'alvo': TARGET_COL,
    'protocolo': 'walk-forward 1-step-ahead, estado sequencial',
    'n_previsoes': int(len(df_wf)),
    'data_inicio_teste': str(df_wf['date'].min().date()),
    'data_fim_teste': str(df_wf['date'].max().date()),
    'MAE': float(mae),
    'RMSE': float(rmse),
    'MAPE': float(mape),
    'R2': float(r2),
    'Bias': float(bias),
    'Std_residuo': float(residuals.std()),
    'tempo_total_s': float(elapsed_seconds),
    'random_state': RANDOM_STATE,
    'order': SARIMAX_ORDER,
    'seasonal_order': SARIMAX_SEASONAL_ORDER,
    'maxiter': MAXITER
}
with (OUTPUT_DIR / 'sarimax_metricas_walkforward.json').open('w', encoding='utf-8') as f:
    json.dump(metricas_export, f, indent=2, ensure_ascii=False)
print('Arquivos exportados em', OUTPUT_DIR)
print('  sarimax_previsoes_walkforward.csv — previsoes, reais e residuos')
print('  sarimax_metricas_walkforward.json — sumario completo de metricas')


Arquivos exportados em C:\Users\raqueltolomei-ieg\OneDrive - Instituto Germinare\Área de Trabalho\3 ano\Séries Temporais\PLSRegretion\pls-regration-comparation\analyses\grupo1\02_modelos\SARIMAX
  sarimax_previsoes_walkforward.csv — previsoes, reais e residuos
  sarimax_metricas_walkforward.json — sumario completo de metricas
